# 02 — The volatility surface, and how it has moved

Notebook 01 fitted one smile per expiry on a single day. This one does two things with that:

1. **Builds a surface**: one object you can ask for the implied vol of *any* strike and maturity, not only the listed ones.
2. **Follows it through time**: every session since collection started, reduced to a few numbers per day (ATM vol and skew at fixed tenors).

Set `REPO`, then Runtime → Run all.

In [ ]:
REPO = "aankit1729/spcx-vol-surface"
RISK_FREE = 0.042          # 3-month T-bill, about 4.2% over the whole sample

import sys, subprocess, shutil
shutil.rmtree("repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", "repo"], check=True)
sys.path.insert(0, "repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from volsurface import surface

COL = {"SPCX": "#2a78d6", "SPY": "#eb6834", "QQQ": "#1baf7a"}      # one colour per ticker, everywhere
INK, MUTED, GRID = "#1a1a19", "#6b6a65", "#e6e5e0"
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": .6, "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
                     "axes.titlesize": 10, "axes.titleweight": "medium", "axes.titlelocation": "left"})
print("ready")

## 1. Build today's surface

`build_surface` runs the whole pipeline on one raw snapshot: clean the quotes, fit an SVI curve to each expiry, stack the curves. The table has one row per expiry — `rmse_bp` is the fit error in basis points of vol, and the two `_ok` columns are the arbitrage checks.

In [ ]:
snaps = surface.list_snapshots("repo/data/raw", "SPCX")       # one snapshot per trading session
raw = pd.read_parquet(snaps["path"].iloc[-1])
surf = surface.build_surface(raw, r=RISK_FREE)

print(f"{surf.ticker}  session {surf.session.date()}  spot {surf.spot:.2f}  {len(surf.slices)} expiries")
surf.slices[["expiry", "days", "F", "n", "atm_vol", "rmse_bp", "butterfly_ok", "calendar_ok"]].round(3)

## 2. Ask it questions

Between listed expiries the surface interpolates *total variance* (σ²T) linearly in time at fixed log-moneyness — the standard rule, because it cannot create calendar arbitrage that the slices themselves do not have.

In [ ]:
T = 0.25                                       # three months, not a listed expiry
print(f"forward for T = 3m:                 {surf.forward(T):.2f}")
print(f"ATM vol:                            {surf.atm_vol(T):.1%}")
print(f"vol of the {0.8*surf.spot:.0f} strike (20% below):  {surf.strike_vol(0.8*surf.spot, T):.1%}")
print(f"vol at log-moneyness k = +0.10:     {surf.iv(0.10, T):.1%}")
print(f"25-delta put / call vol:            {surf.delta_vol(.25, T, -1):.1%} / {surf.delta_vol(.25, T, +1):.1%}")
print(f"25-delta skew (put minus call):     {surf.skew_25d(T)*100:+.1f} vol points")
print()
print("vol grid: rows = days to expiry, columns = strike / forward")
(surf.grid(moneyness=[0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3], days=[14, 30, 91, 182, 365, 730]) * 100).round(1)

In [ ]:
g = surf.grid(moneyness=np.linspace(0.6, 1.4, 81)) * 100
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.25, 1]})

im = ax[0].pcolormesh(g.columns, np.arange(len(g.index) + 1), g.values, cmap="Blues", shading="flat")
step = max(1, len(g.index) // 8)
ax[0].set_yticks(np.arange(len(g.index))[::step] + .5, [f"{d:.0f}" for d in g.index[::step]])
ax[0].grid(False); ax[0].axvline(1, color="white", lw=1)
ax[0].set_xlabel("strike / forward"); ax[0].set_ylabel("days to expiry")
ax[0].set_title(f"{surf.ticker} implied vol surface, {surf.session.date()}")
fig.colorbar(im, ax=ax[0], label="implied vol (%)")

m = np.linspace(0.7, 1.3, 121)
for (name, days), shade in zip(surface.TENORS.items(), ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]):
    v = surf.iv(np.log(m), days / 365.25) * 100
    ax[1].plot(m, v, color=shade)
    ax[1].annotate(name, (m[-1], v[-1]), xytext=(4, 0), textcoords="offset points", va="center", color=MUTED)
ax[1].axvline(1, color=GRID, lw=1)
ax[1].set_xlabel("strike / forward"); ax[1].set_ylabel("implied vol (%)"); ax[1].set_title("Smile at fixed tenors")
plt.tight_layout(); plt.show()

## 3. Every session so far

Fitting takes about ten seconds per ticker per day, so the daily job does it once and stores the fitted parameters in `data/surfaces/`. Loading that table is instant.

Listed expiries get one day shorter every day, so "the December expiry" is not comparable from one week to the next. `summarise_history` reads each day's surface at **fixed tenors** (1m, 3m, 6m, 1y) instead, which is what makes a time series.

In [ ]:
hist = {t: pd.read_parquet(f"repo/data/surfaces/{t}.parquet") for t in ["SPCX", "SPY", "QQQ"]}
summ = {t: surface.summarise_history(h) for t, h in hist.items()}
for t, s in summ.items():
    print(f"{t}: {len(s)} sessions, {s.index.min().date()} to {s.index.max().date()}")
(summ["SPCX"][["spot", "atm_1m", "atm_3m", "atm_1y", "skew_1m", "skew_3m", "skew_1y"]].tail(8)
    .mul([1, 100, 100, 100, 100, 100, 100]).round(2))

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
s = summ["SPCX"]
ax[0].plot(s.index, s["spot"], color=COL["SPCX"]); ax[0].set_title("SPCX share price ($)")

for t in ["SPCX", "SPY", "QQQ"]:
    for a, col in ((ax[1], "atm_1m"), (ax[2], "skew_1m")):
        y = summ[t][col] * 100
        a.plot(y.index, y, color=COL[t])
        a.annotate(t, (y.index[-1], y.iloc[-1]), xytext=(5, 0), textcoords="offset points",
                   va="center", color=MUTED)
ax[1].set_title("1-month ATM implied vol (%)")
ax[2].set_title("1-month 25-delta skew: put vol minus call vol (vol points). Below zero = calls dearer")
ax[2].axhline(0, color=INK, lw=.8)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 4. The same smile, first session and latest

A model-free look at what changed: SPCX's 1-month smile on the first and the most recent session, against moneyness so the two share an axis even though the share price moved.

In [ ]:
surfs = surface.surfaces_from_history(hist["SPCX"])
first, last = min(surfs), max(surfs)
m = np.linspace(0.75, 1.25, 101)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for day, shade in ((first, "#86b6ef"), (last, "#0d366b")):
    v = surfs[day].iv(np.log(m), 30 / 365.25) * 100
    ax.plot(m, v, color=shade)
    ax.annotate(f"{day.date()}", (m[0], v[0]), xytext=(-4, 0), textcoords="offset points",
                ha="right", va="center", color=MUTED)
ax.axvline(1, color=GRID, lw=1); ax.set_xlim(0.62, 1.27)
ax.set_xlabel("strike / forward"); ax.set_ylabel("implied vol (%)"); ax.set_title("SPCX 1-month smile")
plt.tight_layout(); plt.show()

## 5. How good are the fits, day by day?

Before trusting any time series above, check the fits behind it. Median fit error per session, and how many expiries failed an arbitrage check.

In [ ]:
q = pd.DataFrame({t: h.groupby("session")["rmse_bp"].median() for t, h in hist.items()})
fig, ax = plt.subplots(figsize=(9, 3))
for t in q:
    ax.plot(q.index, q[t], color=COL[t])
    ax.annotate(t, (q.index[-1], q[t].iloc[-1]), xytext=(5, 0), textcoords="offset points", va="center", color=MUTED)
ax.set_ylim(0); ax.set_title("Median SVI fit error per session (bp of vol)")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

for t, h in hist.items():
    print(f"{t}: {len(h)} expiry fits | butterfly flags {(~h['butterfly_ok']).sum()} | "
          f"calendar flags {(~h['calendar_ok']).sum()} | fits worse than 50 bp: {(h['rmse_bp'] > 50).sum()}")

## 6. What to write down

1. When did SPCX's skew change sign, and what was the share price doing at the time?
2. Did ATM vol and skew move together, or did skew move while the level stayed put?
3. What happened to the 1-month vol around the earnings date, compared with the 1-year?
4. Did SPY and QQQ do anything similar over the same weeks? If not, the move is specific to SPCX.

**Next:** `03_variance_risk_premium` — compare the 1-month implied vol on each date with the vol the share price then actually realised.